# Boucle neuromodulée — 3 étapes biologiques

On ferme la boucle neuromodulée du système Blob + Predictive Coding :

```
[ Chiffre ] → Physarum Synaptique (Flux) ◄──┐
                   │                          │ Signal Neuromodulateur (Surprise S)
                   ▼                          │ - Contrôle plasticité η(S)
             [ Signature z ]                  │ - Contrôle relaxation N_iter(S)
                   │                          │
                   ▼                          │
        [ Predictive Coding ε = z - ẑ ] ──────┘
                   │
                   ▼
   [ Tuyaux + Inhibition Latérale ] → Décision
```

## Les 3 étapes
1. **Plasticité gérée par la surprise** (3-Factor Hebbian) : `η(S) = η_base + β·S`
   — la surprise module le taux d'apprentissage du Physarum.
2. **Évaluation métabolique temporelle** : `N_iter(S) = N_min + ⌊α·S⌋`
   — plus de surprise = plus d'itérations de relaxation (arousal).
3. **Inhibition latérale inter-tuyaux** : `softmax(A/τ)` — compétition corticale.

## 0. Imports

In [1]:
# Boucle neuromodulée — plasticité gérée par la surprise
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from recherche_agi import (load_mnist, train_readout, NeuromodulatedReservoir,
    lateral_inhibition, metabolic_n_iter, surprise_eta, SynapticReservoir)

## 1. Données : MNIST

In [2]:
train_set, test_set = load_mnist()
print("Train :", len(train_set), "| Test :", len(test_set))

Train : 60000 | Test : 10000


## 2. Étape 1 — Plasticité gérée par la surprise

In [3]:
# η(S) = η_base + β·S : le taux d'apprentissage monte avec la surprise
print("=== η(S) : plasticité modulée par la surprise ===")
for S in [0.0, 0.2, 0.5, 0.8, 1.0]:
    print(f"  S={S:.1f} → η={surprise_eta(S, eta_base=0.1, beta=0.5):.3f}")
print("\nFaible surprise : η≈0.1 (rigide, protège la mémoire)")
print("Forte surprise  : η≈0.6 (liquide, reconfigure vite)")

=== η(S) : plasticité modulée par la surprise ===
  S=0.0 → η=0.100
  S=0.2 → η=0.200
  S=0.5 → η=0.350
  S=0.8 → η=0.500
  S=1.0 → η=0.600

Faible surprise : η≈0.1 (rigide, protège la mémoire)
Forte surprise  : η≈0.6 (liquide, reconfigure vite)


## 3. Étape 2 — Évaluation métabolique temporelle

In [4]:
# N_iter(S) = N_min + floor(α·S) : plus de surprise = plus de calcul (arousal)
print("=== N_iter(S) : relaxation métabolique ===")
for S in [0.0, 0.2, 0.5, 0.8, 1.0]:
    print(f"  S={S:.1f} → N_iter={metabolic_n_iter(S, n_min=5, alpha=40, n_max=50)}")
print("\nChiffre banal : ~5 itérations (économie d'énergie)")
print("Forme nouvelle : ~45 itérations (équilibre complexe)")

=== N_iter(S) : relaxation métabolique ===
  S=0.0 → N_iter=5
  S=0.2 → N_iter=13
  S=0.5 → N_iter=25
  S=0.8 → N_iter=37
  S=1.0 → N_iter=45

Chiffre banal : ~5 itérations (économie d'énergie)
Forme nouvelle : ~45 itérations (équilibre complexe)


## 4. Étape 3 — Inhibition latérale inter-tuyaux

In [5]:
# Compétition corticale : un tuyau actif étouffe les voisins
print("=== Inhibition latérale sur les activations des tuyaux ===")
A = np.array([0.8, 0.3, 0.2, 0.1])   # activations de 4 tuyaux (ex: 8, 3, 5, 7)
print(f"  Activations brutes : {A}")
print(f"  softmax(τ=0.5)     : {np.round(lateral_inhibition(A, 0.5, 'softmax'), 3)}")
print(f"  soustractive       : {np.round(lateral_inhibition(A, 0.5, 'subtractive'), 3)}")
print("\nLe tuyau dominant (0.8) écrase les hésitants → décision nette (WTA doux)")

=== Inhibition latérale sur les activations des tuyaux ===
  Activations brutes : [0.8 0.3 0.2 0.1]
  softmax(τ=0.5)     : [0.522 0.192 0.157 0.129]
  soustractive       : [0.6 0.  0.  0. ]

Le tuyau dominant (0.8) écrase les hésitants → décision nette (WTA doux)


## 5. Pipeline neuromodulé complet

Le `NeuromodulatedReservoir` boucle : il estime la surprise S via un prédicteur,
puis module sa propre dynamique (plasticité η(S) + relaxation N_iter(S)) avant de
produire la signature z. Comparé au réservoir de base.

In [6]:
def extract(reservoir, dataset, n):
    X, y = [], []
    cnt = [0]*10
    for i in range(len(dataset)):
        l = int(dataset[i][1])
        if cnt[l] >= n//10: continue
        X.append(reservoir.signature(dataset[i][0].squeeze().numpy()))
        y.append(l); cnt[l] += 1
        if sum(cnt) >= n: break
    return np.array(X), np.array(y)

# Réservoir neuromodulé (predictor = identité pour la démo, S=0)
neu = NeuromodulatedReservoir(axes=('top_down','left_right'), n_zones=32, downscale=8,
                              predictor=lambda z: z, n_max=20)
Xtr, ytr = extract(neu, train_set, 100)
print(f"Signatures neuromodulées : {Xtr.shape}")

ro = train_readout(Xtr, ytr, n_classes=10, epochs=50)
with torch.no_grad():
    acc = (ro(torch.tensor(Xtr, dtype=torch.float32)).argmax(1) == torch.tensor(ytr)).float().mean().item()
print(f"Acc couche lue neuromodulée : {acc:.3f}")

Signatures neuromodulées : (100, 64)


Acc couche lue neuromodulée : 0.520


## 6. Synthèse

In [7]:
print("=== SYNTHÈSE DE LA BOUCLE NEUROMODULÉE ===")
print("Les 3 étapes ferment la boucle Blob + PC :")
print("  1. Surprise S = ||z - ẑ|| (erreur de prédiction du PC)")
print("  2. S module la plasticité η(S) du Physarum (3-facteur Hebbien)")
print("  3. S module la relaxation N_iter(S) (arousal métabolique)")
print("  4. Inhibition latérale entre tuyaux → décision nette (WTA doux)")
print()
print("Le système adapte SA PROPRE dynamique selon l'inattendu :")
print("  - connu → rigide, économe (protège la mémoire)")
print("  - nouveau → plastique, profond (reconfigure vite)")

=== SYNTHÈSE DE LA BOUCLE NEUROMODULÉE ===
Les 3 étapes ferment la boucle Blob + PC :
  1. Surprise S = ||z - ẑ|| (erreur de prédiction du PC)
  2. S module la plasticité η(S) du Physarum (3-facteur Hebbien)
  3. S module la relaxation N_iter(S) (arousal métabolique)
  4. Inhibition latérale entre tuyaux → décision nette (WTA doux)

Le système adapte SA PROPRE dynamique selon l'inattendu :
  - connu → rigide, économe (protège la mémoire)
  - nouveau → plastique, profond (reconfigure vite)
